In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import json
import random
import matplotlib.pyplot as plt
import wandb
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import accuracy_score, classification_report
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
random.seed(42)

train_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
test_data  = fetch_20newsgroups(subset='test',  remove=('headers', 'footers', 'quotes'))

N_EVAL  = 200
indices = random.sample(range(len(test_data.data)), N_EVAL)
eval_texts  = [test_data.data[i]   for i in indices]
eval_labels = [test_data.target[i] for i in indices]

CLASS_NAMES = train_data.target_names
print(f'Evaluating on {N_EVAL} samples across {len(CLASS_NAMES)} classes')

In [ ]:
MODEL_ID   = 'meta-llama/Meta-Llama-3-8B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

print('Loading Llama-3 (this takes a few minutes)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
model.eval()
print('Model loaded.')

In [ ]:
def ask_llama(prompt, max_new_tokens=20):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                do_sample=False, pad_token_id=tokenizer.eos_token_id)
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def match_class(response, class_names):
    response = response.lower()
    for i, name in enumerate(class_names):
        if name.lower() in response or response in name.lower():
            return i
    return 0   # fallback to first class if no match


def run_experiment(prompts, labels, class_names, run_name):
    wandb.init(project='nalapro-20newsgroups', name=run_name,
               config={'n_eval': N_EVAL}, reinit=True)
    preds = []
    for i, prompt in enumerate(prompts):
        response = ask_llama(prompt)
        preds.append(match_class(response, class_names))
        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(prompts)} done...')
    acc = accuracy_score(labels, preds)
    wandb.log({'accuracy': acc})
    wandb.finish()
    return preds, acc

print('Helper functions ready.')

In [ ]:
CLASS_LIST = '\n'.join(f'- {name}' for name in CLASS_NAMES)

def zero_shot_prompt(text):
    return f"""Classify the following text into exactly one of these categories:
{CLASS_LIST}

Text: {text[:300]}

Reply with only the category name.
Category:"""

print(f'Running zero-shot on {N_EVAL} samples...')
preds_zero, acc_zero = run_experiment(
    [zero_shot_prompt(t) for t in eval_texts],
    eval_labels, CLASS_NAMES, run_name='llama3-zero-shot'
)
print(f'\nZero-Shot Accuracy: {acc_zero:.4f}')
print(classification_report(eval_labels, preds_zero, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# Pick one training example per class
EXAMPLE_CLASSES = [0, 7, 14]

examples = {}
for text, label in zip(train_data.data, train_data.target):
    if label in EXAMPLE_CLASSES and label not in examples:
        examples[label] = text[:150]
    if len(examples) == len(EXAMPLE_CLASSES):
        break

examples_block = '\n\n'.join(
    f'Text: {examples[i]}\nCategory: {CLASS_NAMES[i]}'
    for i in EXAMPLE_CLASSES
)

def few_shot_prompt(text):
    return f"""Classify text into one of these categories:
{CLASS_LIST}

Examples:
{examples_block}

Text: {text[:200]}
Reply with only the category name.
Category:"""


print(f'Running few-shot on {N_EVAL} samples...')
preds_few, acc_few = run_experiment(
    [few_shot_prompt(t) for t in eval_texts],
    eval_labels, CLASS_NAMES, run_name='llama3-few-shot'
)
print(f'\nFew-Shot Accuracy: {acc_few:.4f}')
print(classification_report(eval_labels, preds_few, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
acc_w2v   = 0.586
acc_tfidf = 0.636
acc_lsa   = 0.615
acc_bert     = 0.698
acc_bert_mlm = 0.710

labels = ['Word2Vec', 'TF-IDF', 'TF-IDF+LSA', 'BERT', 'BERT+MLM', 'Llama3\nZero-Shot', 'Llama3\nFew-Shot']
accs   = [acc_w2v, acc_tfidf, acc_lsa, acc_bert, acc_bert_mlm, acc_zero, acc_few]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#9467BD', '#8C564B', '#E377C2']

plt.figure(figsize=(13, 5))
bars = plt.bar(labels, accs, color=colors, width=0.6)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f'{acc:.3f}', ha='center', fontweight='bold')
plt.ylim(0, 1.05)
plt.ylabel('Test Accuracy')
plt.title('Final Comparison: All Methods', fontsize=14)
plt.tight_layout()
plt.savefig('final_comparison_all.png', dpi=150)
plt.show()